<a href="https://colab.research.google.com/github/escriber880/phosphopower-landing/blob/main/notebook83c0c97442.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import sys
import site
import shutil
import subprocess
from pathlib import Path

# This cell must run in the SAME GPU Colab runtime as the YuE2 program.
if not shutil.which('nvidia-smi'):
    raise RuntimeError(
        'No GPU runtime detected: nvidia-smi is missing. In Colab select '
        'Runtime > Change runtime type > GPU (T4 if available), save, reconnect, '
        'then run this cell again.'
    )

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout[:2500] or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError('nvidia-smi failed; the GPU is not ready.')

server = Path('/content/yue2.cpp/build/yue-server')

# This yue-server binary requires CUDA 13 runtime libraries.
print('Installing CUDA 13 runtime and cuBLAS libraries...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'nvidia-cuda-runtime', 'nvidia-cublas'
], check=True)


def cuda_library_dirs():
    found = []
    bases = set(site.getsitepackages() + [site.getusersitepackages()])
    patterns = (
        'libcudart.so.13*',
        'libcublas.so.13*',
        'libcublasLt.so.13*'
    )
    for base in bases:
        nvidia_dir = Path(base) / 'nvidia'
        if not nvidia_dir.is_dir():
            continue
        for pattern in patterns:
            for library in nvidia_dir.rglob(pattern):
                found.append(str(library.parent))
    return list(dict.fromkeys(found))


lib_dirs = cuda_library_dirs()
if not any((Path(p) / 'libcudart.so.13').exists() for p in lib_dirs):
    matches = []
    for base in set(site.getsitepackages() + [site.getusersitepackages()]):
        nvidia_dir = Path(base) / 'nvidia'
        if nvidia_dir.is_dir():
            matches.extend(str(p) for p in nvidia_dir.rglob('libcudart.so.13*'))
    if not matches:
        raise RuntimeError('Installed package did not provide libcudart.so.13.')

old_paths = [p for p in os.environ.get('LD_LIBRARY_PATH', '').split(':') if p]
new_paths = [str(server.parent)] + lib_dirs + old_paths
os.environ['LD_LIBRARY_PATH'] = ':'.join(dict.fromkeys(new_paths))

print('LD_LIBRARY_PATH updated.')
if server.is_file():
    check = subprocess.run(
        ['ldd', str(server)], capture_output=True, text=True, env=os.environ
    )
    print(check.stdout or check.stderr)
    missing = [line for line in check.stdout.splitlines() if 'not found' in line]
    if missing:
        raise RuntimeError('Some shared libraries are still missing; copy the ldd output above.')

    help_result = subprocess.run(
        [str(server), '--help'], cwd=server.parent.parent,
        capture_output=True, text=True, env=os.environ, timeout=20
    )
    print((help_result.stdout + '\n' + help_result.stderr)[:3000])
    if help_result.returncode != 0:
        raise RuntimeError('yue-server still cannot start; copy the output above.')
else:
    print('yue-server is not installed yet; run the main YuE2 setup program next.')

print('CUDA runtime setup complete. Run the YuE2 program in this same runtime.')

Fri Oct 23 03:09:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
import sys
import site
import subprocess
from pathlib import Path

# Confirma que o Colab está com GPU ativa
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError(
        "GPU não detectada. Selecione uma GPU em "
        "Runtime > Change runtime type e execute esta célula novamente."
    )

# Instala o runtime CUDA 13 e cuBLAS
# Para CUDA 13, os nomes corretos não levam o sufixo -cu13.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "nvidia-cuda-runtime", "nvidia-cublas"
])

# Adiciona as pastas das bibliotecas CUDA ao ambiente deste kernel
lib_dirs = set()
bases = set(site.getsitepackages() + [site.getusersitepackages()])

for base in bases:
    nvidia_dir = Path(base) / "nvidia"
    if nvidia_dir.is_dir():
        for name in (
            "libcudart.so.13",
            "libcublas.so.13",
            "libcublasLt.so.13",
        ):
            lib_dirs.update(str(p.parent) for p in nvidia_dir.rglob(name))

if not any((Path(d) / "libcudart.so.13").is_file() for d in lib_dirs):
    raise RuntimeError(
        "O pacote instalou, mas libcudart.so.13 não foi encontrado."
    )

old_dirs = [
    p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p
]
os.environ["LD_LIBRARY_PATH"] = ":".join(sorted(lib_dirs) + old_dirs)

# Verifica se o binário encontra as bibliotecas
server = Path("/content/yue2.cpp/build/yue-server")
if server.is_file():
    check = subprocess.run(
        ["ldd", str(server)],
        capture_output=True,
        text=True,
        env=os.environ
    )
    print(check.stdout or check.stderr)
    if "not found" in check.stdout:
        raise RuntimeError(
            "Ainda há bibliotecas ausentes. Envie a saída do ldd acima."
        )

print("Preparação CUDA concluída. Agora execute a célula principal do YuE2.")

Fri Oct 23 03:09:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#@title YuE2 Colab - debugado
# Cole somente este codigo em uma celula do Colab. Nao copie os delimitadores do chat.

import os, re, sys, time, shutil, zipfile, subprocess
from pathlib import Path
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from IPython.display import display, HTML

ROOT = Path('/content/yue2.cpp')
BUILD = ROOT / 'build'
MODELS = ROOT / 'models'
SERVER = BUILD / 'yue-server'
MODEL = MODELS / 'YuE2-3B-Q8_0.gguf'
VAE = MODELS / 'YuE2-Vae-F32.gguf'
SERVER_LOG = ROOT / 'server.log'
TUNNEL_LOG = ROOT / 'tunnel.log'
PORT = 8087


def run(args, cwd=None):
    return subprocess.run(args, cwd=cwd, check=True)


def tail(path, count=60):
    path = Path(path)
    print('\n--- ' + path.name + ' ---')
    if path.exists():
        lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(lines[-count:]) if lines else '(log vazio)')
    else:
        print('(arquivo nao encontrado)')


def status(url, timeout=3):
    try:
        req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(req, timeout=timeout) as response:
            return response.status
    except HTTPError as err:
        return err.code
    except (URLError, TimeoutError, OSError):
        return None


# Para processos antigos desta sessao.
for name in ('server_proc', 'tunnel_proc', 'server_process', 'tunnel_process'):
    old = globals().get(name)
    if isinstance(old, subprocess.Popen) and old.poll() is None:
        old.terminate()
        try:
            old.wait(timeout=5)
        except subprocess.TimeoutExpired:
            old.kill()
try:
    subprocess.run(['pkill', '-TERM', '-x', 'yue-server'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except FileNotFoundError:
    pass

# 1. Repositorio e binario.
if not ROOT.is_dir():
    print('Clonando yue2.cpp...')
    run(['git', 'clone', '--recurse-submodules',
         'https://github.com/ServeurpersoCom/yue2.cpp', str(ROOT)])
BUILD.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

if not SERVER.is_file():
    archive_path = Path('/content/yue2_t4_binaries.zip')
    archive_url = ('https://github.com/Harzur/YuE2-Q8_0-Colab-T4/releases/'
                   'download/v1.0-t4/yue2_t4_binaries.zip')
    if not zipfile.is_zipfile(archive_path):
        if archive_path.exists():
            archive_path.unlink()
        print('Baixando binarios T4...')
        run(['wget', '-q', '--show-progress', archive_url, '-O', str(archive_path)])
    print('Extraindo binarios...')
    with zipfile.ZipFile(archive_path) as archive:
        files = [x for x in archive.infolist()
                 if x.filename.startswith('build/') and not x.is_dir()]
        if not files:
            raise RuntimeError('O ZIP nao tem arquivos dentro de build/.')
        for item in files:
            target = BUILD / Path(item.filename).relative_to('build')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
if not SERVER.is_file():
    raise FileNotFoundError('Binario yue-server nao encontrado: ' + str(SERVER))
SERVER.chmod(0o755)

# 2. CLI hf e modelos requeridos.
hf = shutil.which('hf')
if not hf:
    print('Instalando ferramenta hf...')
    run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub'])
    hf = shutil.which('hf')
if not hf:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'hf'])
    hf = shutil.which('hf')
if not hf:
    raise RuntimeError('Comando hf nao encontrado apos a instalacao.')


def get_model(path, filename, min_size):
    if path.is_file() and path.stat().st_size >= min_size:
        print('Modelo pronto: ' + filename)
        return
    if path.exists():
        path.unlink()
    print('Baixando ' + filename + '...')
    run([hf, 'download', 'Serveurperso/YuE2-GGUF', filename,
         '--local-dir', str(MODELS)], cwd=ROOT)
    if not path.is_file() or path.stat().st_size < min_size:
        raise RuntimeError('Download incompleto: ' + str(path))


get_model(MODEL, 'YuE2-3B-Q8_0.gguf', 1_000_000_000)
get_model(VAE, 'YuE2-Vae-F32.gguf', 100_000_000)

# 3. Servidor.
server_cmd = [str(SERVER), '--host', '0.0.0.0', '--port', str(PORT),
              '--model', 'models/YuE2-3B-Q8_0.gguf',
              '--vae', 'models/YuE2-Vae-F32.gguf']

# Usa o contexto padrao do modelo (24576).
# Nao force --max-seq 8192: o budget padrao da WebUI e 9000.

print('Iniciando yue-server...')
with open(SERVER_LOG, 'w', encoding='utf-8') as log:
    server_proc = subprocess.Popen(server_cmd, cwd=ROOT,
                                   stdout=log, stderr=subprocess.STDOUT)

print('Aguardando resposta HTTP local (ate 6 minutos)...')
ready = False
start = time.monotonic()
while time.monotonic() - start < 360:
    if server_proc.poll() is not None:
        print('yue-server encerrou, codigo:', server_proc.returncode)
        tail(SERVER_LOG)
        raise RuntimeError('Falha ao iniciar yue-server.')
    code = status('http://127.0.0.1:' + str(PORT) + '/health')
    if code is not None and 200 <= code < 400:
        ready = True
        break
    elapsed = int(time.monotonic() - start)
    if elapsed % 10 < 2:
        print('Carregando modelo...', elapsed, 's; HTTP:', code)
    time.sleep(2)
if not ready:
    server_proc.terminate()
    tail(SERVER_LOG)
    raise TimeoutError('WebUI local nao respondeu; Cloudflare nao foi iniciado.')
print('Servidor local respondeu. Iniciando Cloudflare...')

# 4. Cloudflare Tunnel.
choices = [ROOT / 'cloudflared-linux-amd64', Path('/content/cloudflared'),
           Path('/content/cloudflared-linux-amd64')]
cloud = next((p for p in choices if p.is_file()), None)
if cloud is None or cloud.stat().st_size < 1_000_000:
    cloud = ROOT / 'cloudflared-linux-amd64'
    run(['wget', '-q',
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
         '-O', str(cloud)])
if cloud.stat().st_size < 1_000_000:
    server_proc.terminate()
    raise RuntimeError('Download do cloudflared incompleto.')
cloud.chmod(0o755)

with open(TUNNEL_LOG, 'w', encoding='utf-8') as log:
    tunnel_proc = subprocess.Popen([str(cloud), 'tunnel', '--url',
                                    'http://127.0.0.1:' + str(PORT)],
                                   cwd=ROOT, stdout=log,
                                   stderr=subprocess.STDOUT)

url = None
start = time.monotonic()
while time.monotonic() - start < 60:
    if tunnel_proc.poll() is not None:
        tail(TUNNEL_LOG)
        server_proc.terminate()
        raise RuntimeError('cloudflared encerrou antes de criar URL.')
    if TUNNEL_LOG.exists():
        found = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',
                          TUNNEL_LOG.read_text(encoding='utf-8', errors='replace'))
        if found:
            url = found.group(0)
            break
    time.sleep(1)
if not url:
    tail(TUNNEL_LOG)
    server_proc.terminate()
    raise TimeoutError('Cloudflare nao forneceu URL em 60 segundos.')

# Espera o link publico responder antes de mostra-lo.
public_code = None
for attempt in range(15):
    public_code = status(url)
    if public_code is not None and 200 <= public_code < 400:
        break
    print('Aguardando tunel...', attempt + 1, '/15; HTTP:', public_code)
    time.sleep(2)
if public_code is None or public_code >= 400:
    print('Link ainda nao confirmado; log do tunel:')
    tail(TUNNEL_LOG, 40)
else:
    print('Link publico respondeu HTTP', public_code)

display(HTML('<a href="' + url + '" target="_blank">Abrir YuE2 Studio</a><br>' + url))
print('Mantenha esta celula rodando. O Colab encerra servidor/tunel ao desconectar.')

# 5. Mantem processos vivos e mostra eventos relevantes do log.
pos = 0
try:
    while True:
        if server_proc.poll() is not None:
            print('yue-server parou, codigo:', server_proc.returncode)
            tail(SERVER_LOG, 100)
            if tunnel_proc.poll() is None:
                tunnel_proc.terminate()
            break
        if tunnel_proc.poll() is not None:
            print('cloudflared parou, codigo:', tunnel_proc.returncode)
            tail(TUNNEL_LOG, 80)
            if server_proc.poll() is None:
                server_proc.terminate()
            break
        with open(SERVER_LOG, 'r', encoding='utf-8', errors='replace') as log:
            log.seek(pos)
            lines = log.readlines()
            pos = log.tell()
        for line in lines:
            text = line.rstrip()
            if text and re.search(r'(synth|generat|compos|step|semantic|NAR|VAE|error|fail|cuda|saved|request)', text, re.I):
                print('[server]', text[:500])
        time.sleep(3)
except KeyboardInterrupt:
    print('Encerrando servidor e tunel...')
    for proc in (server_proc, tunnel_proc):
        if proc.poll() is None:
            proc.terminate()

Modelo pronto: YuE2-3B-Q8_0.gguf
Modelo pronto: YuE2-Vae-F32.gguf
Iniciando yue-server...
Aguardando resposta HTTP local (ate 6 minutos)...
Carregando modelo... 0 s; HTTP: None
Servidor local respondeu. Iniciando Cloudflare...
Aguardando tunel... 1 /15; HTTP: None
Aguardando tunel... 2 /15; HTTP: None
Aguardando tunel... 3 /15; HTTP: None
Aguardando tunel... 4 /15; HTTP: None
Aguardando tunel... 5 /15; HTTP: None
Aguardando tunel... 6 /15; HTTP: None
Aguardando tunel... 7 /15; HTTP: None
Aguardando tunel... 8 /15; HTTP: None
Aguardando tunel... 9 /15; HTTP: None
Aguardando tunel... 10 /15; HTTP: None
Aguardando tunel... 11 /15; HTTP: None
Aguardando tunel... 12 /15; HTTP: None
Aguardando tunel... 13 /15; HTTP: None
Aguardando tunel... 14 /15; HTTP: None
Aguardando tunel... 15 /15; HTTP: None
Link ainda nao confirmado; log do tunel:

--- tunnel.log ---
2026-10-23T04:40:29Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-23T04:40

Mantenha esta celula rodando. O Colab encerra servidor/tunel ao desconectar.
[server] ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
[server] [Load] KV backend: CUDA0 (CPU threads: 1)
[server] [Load] LM backend: CUDA0 (CPU threads: 1)
[server] [AR] Score song 0: end token at step 1621
[server] [AR] Score: 1621 tokens over 1 songs, 1622 steps, 18.9 s (11.7 ms/step)
[server] [AR] Semantic song 0: 1992 tokens prefilled, 450 ms
[server] [AR] Semantic prefill: 450 ms, CFG=1.00, top_k=100, budget=9000, songs=1, batch=1
[server] [AR] Semantic 0/9000
[server] [AR] Semantic 100/9000
[server] [AR] Semantic 200/9000
[server] [AR] Semantic 300/9000
[server] [AR] Semantic 400/9000
[server] [AR] Semantic 500/9000
[server] [AR] Semantic 600/9000
[server] [AR] Semantic 700/9000
[server] [AR] Semantic 800/9000
[server] [AR] Semantic 900/9000
[server] [AR] Semantic 1000/9000
[server] [AR] Semantic 1100/9000
[server] [AR] Semantic 1200/9000
[server] [AR] Semantic 1300/9000
[server] [AR] Se

In [14]:
from pathlib import Path

p = Path("/content/yue2.cpp/server.log")
if not p.exists():
    print("Não encontrei:", p)
else:
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    print("Total de linhas:", len(lines))
    print("\n".join(lines[-120:]))

Total de linhas: 392
[LM-KV] Allocated 1 sets x 28 layers (4D batched), 896.0 MB
[AR] Score song 0: 364 tokens prefilled, 140 ms
[AR] Score prefill: 140 ms, CFG=1.00, top_k=30, budget=4096, songs=1, batch=1
[AR] Score 0/4096
[AR] Score 100/4096
[AR] Score 200/4096
[AR] Score 300/4096
[AR] Score 400/4096
[AR] Score 500/4096
[AR] Score 600/4096
[AR] Score 700/4096
[AR] Score 800/4096
[AR] Score 900/4096
[AR] Score 1000/4096
[AR] Score 1100/4096
[AR] Score 1200/4096
[AR] Score 1300/4096
[AR] Score 1400/4096
[AR] Score song 0: end token at step 1419
[AR] Score song 0: 1419 tokens
[AR] Score: 1419 tokens over 1 songs, 1420 steps, 17.9 s (12.6 ms/step)
[Prompt] cot=full, songs=1, variations=1, 1 tracks
[AR] FATAL: prefix 1785 + budget 9000 + end exceeds context 8192
[Store] Unload LM (2195.1 MB)
[Server] Job e954ea4a757e841c: {
  "style": "gospel Christian afrobeat",
  "lyrics": "[Title: And Who Am I?]\n\n[Verse 1]\nOh, Lord!\nWhen I look down at myself,\nHow small I am inside\nBefore Your h

#@title YuE2 Colab - debugado

import os, re, sys, time, shutil, zipfile, subprocess
from pathlib import Path
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
from IPython.display import display, HTML

ROOT = Path('/content/yue2.cpp')
BUILD = ROOT / 'build'
MODELS = ROOT / 'models'
SERVER = BUILD / 'yue-server'
MODEL = MODELS / 'YuE2-3B-Q8_0.gguf'
VAE = MODELS / 'YuE2-Vae-F32.gguf'
SERVER_LOG = ROOT / 'server.log'
TUNNEL_LOG = ROOT / 'tunnel.log'
PORT = 8087


def run(args, cwd=None):
    return subprocess.run(args, cwd=cwd, check=True)


def tail(path, count=60):
    path = Path(path)
    print('\n--- ' + path.name + ' ---')
    if path.exists():
        lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(lines[-count:]) if lines else '(log vazio)')
    else:
        print('(arquivo nao encontrado)')


def status(url, timeout=3):
    try:
        req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(req, timeout=timeout) as response:
            return response.status
    except HTTPError as err:
        return err.code
    except (URLError, TimeoutError, OSError):
        return None


# Para processos antigos desta sessao.
for name in ('server_proc', 'tunnel_proc', 'server_process', 'tunnel_process'):
    old = globals().get(name)
    if isinstance(old, subprocess.Popen) and old.poll() is None:
        old.terminate()
        try:
            old.wait(timeout=5)
        except subprocess.TimeoutExpired:
            old.kill()

try:
    subprocess.run(
        ['pkill', '-TERM', '-x', 'yue-server'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
except FileNotFoundError:
    pass


# 1. Repositorio e binario.
if not ROOT.is_dir():
    print('Clonando yue2.cpp...')
    run([
        'git', 'clone', '--recurse-submodules',
        'https://github.com/ServeurpersoCom/yue2.cpp',
        str(ROOT)
    ])

BUILD.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

if not SERVER.is_file():
    archive_path = Path('/content/yue2_t4_binaries.zip')
    archive_url = (
        'https://github.com/Harzur/YuE2-Q8_0-Colab-T4/releases/'
        'download/v1.0-t4/yue2_t4_binaries.zip'
    )

    if not zipfile.is_zipfile(archive_path):
        if archive_path.exists():
            archive_path.unlink()
        print('Baixando binarios T4...')
        run([
            'wget', '-q', '--show-progress',
            archive_url, '-O', str(archive_path)
        ])

    print('Extraindo binarios...')
    with zipfile.ZipFile(archive_path) as archive:
        files = [
            x for x in archive.infolist()
            if x.filename.startswith('build/') and not x.is_dir()
        ]
        if not files:
            raise RuntimeError('O ZIP nao tem arquivos dentro de build/.')

        for item in files:
            target = BUILD / Path(item.filename).relative_to('build')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

if not SERVER.is_file():
    raise FileNotFoundError('Binario yue-server nao encontrado: ' + str(SERVER))

SERVER.chmod(0o755)


# 2. CLI hf e modelos requeridos.
hf = shutil.which('hf')
if not hf:
    print('Instalando ferramenta hf...')
    run([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'huggingface_hub'
    ])
    hf = shutil.which('hf')

if not hf:
    run([
        sys.executable, '-m', 'pip', 'install', '-q', '-U', 'hf'
    ])
    hf = shutil.which('hf')

if not hf:
    raise RuntimeError('Comando hf nao encontrado apos a instalacao.')


def get_model(path, filename, min_size):
    if path.is_file() and path.stat().st_size >= min_size:
        print('Modelo pronto: ' + filename)
        return

    if path.exists():
        path.unlink()

    print('Baixando ' + filename + '...')
    run([
        hf, 'download', 'Serveurperso/YuE2-GGUF', filename,
        '--local-dir', str(MODELS)
    ], cwd=ROOT)

    if not path.is_file() or path.stat().st_size < min_size:
        raise RuntimeError('Download incompleto: ' + str(path))


get_model(MODEL, 'YuE2-3B-Q8_0.gguf', 1_000_000_000)
get_model(VAE, 'YuE2-Vae-F32.gguf', 100_000_000)


# 3. Servidor.
# Os logs vao para arquivo para evitar bloqueio do processo.
server_cmd = [
    str(SERVER), '--host', '0.0.0.0', '--port', str(PORT),
    '--model', 'models/YuE2-3B-Q8_0.gguf',
    '--vae', 'models/YuE2-Vae-F32.gguf'
]

try:
    help_out = subprocess.run(
        [str(SERVER), '--help'],
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=15
    )
    if '--max-seq' in (help_out.stdout or '') + (help_out.stderr or ''):
        server_cmd += ['--max-seq', '8192']
except Exception:
    pass

print('Iniciando yue-server...')
with open(SERVER_LOG, 'w', encoding='utf-8') as log:
    server_proc = subprocess.Popen(
        server_cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT
    )

print('Aguardando resposta HTTP local (ate 6 minutos)...')
ready = False
start = time.monotonic()

while time.monotonic() - start < 360:
    if server_proc.poll() is not None:
        print('yue-server encerrou, codigo:', server_proc.returncode)
        tail(SERVER_LOG)
        raise RuntimeError('Falha ao iniciar yue-server.')

    code = status('http://127.0.0.1:' + str(PORT) + '/')
    if code is not None and 200 <= code < 400:
        ready = True
        break

    elapsed = int(time.monotonic() - start)
    if elapsed % 10 < 2:
        print('Carregando modelo...', elapsed, 's; HTTP:', code)

    time.sleep(2)

if not ready:
    server_proc.terminate()
    tail(SERVER_LOG)
    raise TimeoutError('WebUI local nao respondeu; Cloudflare nao foi iniciado.')

print('Servidor local respondeu. Iniciando Cloudflare...')


# 4. Cloudflare Tunnel.
choices = [
    ROOT / 'cloudflared-linux-amd64',
    Path('/content/cloudflared'),
    Path('/content/cloudflared-linux-amd64')
]
cloud = next((p for p in choices if p.is_file()), None)

if cloud is None or cloud.stat().st_size < 1_000_000:
    cloud = ROOT / 'cloudflared-linux-amd64'
    run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/'
        'download/cloudflared-linux-amd64',
        '-O', str(cloud)
    ])

if cloud.stat().st_size < 1_000_000:
    server_proc.terminate()
    raise RuntimeError('Download do cloudflared incompleto.')

cloud.chmod(0o755)

with open(TUNNEL_LOG, 'w', encoding='utf-8') as log:
    tunnel_proc = subprocess.Popen(
        [
            str(cloud), 'tunnel', '--url',
            'http://127.0.0.1:' + str(PORT)
        ],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT
    )

url = None
start = time.monotonic()

while time.monotonic() - start < 60:
    if tunnel_proc.poll() is not None:
        tail(TUNNEL_LOG)
        server_proc.terminate()
        raise RuntimeError('cloudflared encerrou antes de criar URL.')

    if TUNNEL_LOG.exists():
        found = re.search(
            r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',
            TUNNEL_LOG.read_text(encoding='utf-8', errors='replace')
        )
        if found:
            url = found.group(0)
            break

    time.sleep(1)

if not url:
    tail(TUNNEL_LOG)
    server_proc.terminate()
    raise TimeoutError('Cloudflare nao forneceu URL em 60 segundos.')


# Confirma se o link publico responde antes de mostra-lo.
public_code = None
for attempt in range(15):
    public_code = status(url)
    if public_code is not None and 200 <= public_code < 400:
        break

    print('Aguardando tunel...', attempt + 1, '/15; HTTP:', public_code)
    time.sleep(2)

if public_code is None or public_code >= 400:
    print('Link ainda nao confirmado; log do tunel:')
    tail(TUNNEL_LOG, 40)
else:
    print('Link publico respondeu HTTP', public_code)

display(HTML(
    '<a href="' + url + '" target="_blank">Abrir YuE2 Studio</a><br>' + url
))
print('Mantenha esta celula rodando. O Colab encerra servidor/tunel ao desconectar.')


# 5. Mantem processos vivos e mostra eventos relevantes do log.
pos = 0
try:
    while True:
        if server_proc.poll() is not None:
            print('yue-server parou, codigo:', server_proc.returncode)
            tail(SERVER_LOG, 100)
            if tunnel_proc.poll() is None:
                tunnel_proc.terminate()
            break

        if tunnel_proc.poll() is not None:
            print('cloudflared parou, codigo:', tunnel_proc.returncode)
            tail(TUNNEL_LOG, 80)
            if server_proc.poll() is None:
                server_proc.terminate()
            break

        with open(SERVER_LOG, 'r', encoding='utf-8', errors='replace') as log:
            log.seek(pos)
            lines = log.readlines()
            pos = log.tell()

        for line in lines:
            text = line.rstrip()
            if text and re.search(
                r'(synth|generat|compos|step|semantic|NAR|VAE|error|fail|cuda|saved|request)',
                text,
                re.I
            ):
                print('[server]', text[:500])

        time.sleep(3)

except KeyboardInterrupt:
    print('Encerrando servidor e tunel...')
    for proc in (server_proc, tunnel_proc):
        if proc.poll() is None:
            proc.terminate()